# 03 — Robot Baseline Strategy
Indicators, score generation and corrected portfolio backtest.


In [1]:
from pathlib import Path
import sys
import pandas as pd

PROJECT_ROOT = Path.cwd().parent if not (Path.cwd() / "src").exists() else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

from src.config import StrategyConfig, PortfolioConfig
from src.features import add_indicators
from src.signals import build_market_regime, add_robot_scores
from src.backtest import run_portfolio_backtest
from src.metrics import portfolio_metrics, yearly_performance, monthly_performance


In [2]:
stock_prices = pd.read_parquet(
    PROJECT_ROOT / "data" / "processed" / "bist100_robot_clean.parquet"
)
market_prices = pd.read_parquet(
    PROJECT_ROOT / "data" / "processed" / "xu100_robot_clean.parquet"
)

stock_features = add_indicators(stock_prices)
market_features = add_indicators(market_prices)

market_regime = build_market_regime(market_features)
scored_prices = add_robot_scores(
    stock_features,
    market_regime,
    StrategyConfig(),
)


In [3]:
equity_df, trades_df = run_portfolio_backtest(
    scored_prices,
    StrategyConfig(),
    PortfolioConfig(),
)

metrics = portfolio_metrics(equity_df, trades_df)
display(pd.DataFrame([metrics]))
display(trades_df.head())


,Start_Value,End_Value,Total_Return_%,CAGR_%,Max_Drawdown_%,Profit_Factor,Win_Rate_%,Expectancy_%,Sharpe,Sortino,Calmar,Trade_Count,Outlier_Count,Exposure_%,Average_Open_Positions,Max_Open_Positions,Average_Invested_%
0,500000.0,9.435985e+06,1787.197014,45.986671,-27.915671,2.054777,37.621359,5.219497,2.037607,2.681224,1.647342,824,0,98.557445,7.25863,8,68.100607


,Ticker,Entry_Date,Exit_Date,Entry,Exit,Shares,Return,Return_%,Reason,Is_Outlier,Signal_Score
0,TKFEN.IS,2018-10-19,2018-10-26,18.114150,16.570883,2325,-0.088849,-8.884866,Stop Loss,False,12
1,ANSGR.IS,2018-10-19,2018-11-01,0.844717,0.817866,124229,-0.035652,-3.565183,Stop Loss,False,12
2,SARKY.IS,2018-11-30,2018-12-04,0.600889,0.560020,85806,-0.071735,-7.173507,Stop Loss,False,11
3,SASA.IS,2018-11-28,2018-12-12,0.130286,0.121691,369443,-0.069704,-6.970398,LOW10 Altı,False,12
4,TUPRS.IS,2018-12-10,2018-12-13,10.460171,9.903217,4854,-0.057025,-5.702465,LOW10 Altı,False,11


In [4]:
results_dir = PROJECT_ROOT / "results"
results_dir.mkdir(parents=True, exist_ok=True)

equity_df.to_parquet(results_dir / "baseline_equity.parquet", index=False)
trades_df.to_csv(results_dir / "baseline_trades.csv", index=False)
yearly_performance(equity_df).to_csv(
    results_dir / "baseline_yearly.csv", index=False
)
monthly_performance(equity_df).to_csv(
    results_dir / "baseline_monthly.csv", index=False
)
